In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

books = []

# Scrape first 5 pages
for page in range(1, 6):

    url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    response = requests.get(url)

    soup = BeautifulSoup(response.text, "html.parser")

    # Find all books on the page
    products = soup.find_all("article", class_="product_pod")

    for product in products:

        # Title
        title = product.h3.a["title"]

        # Price
        price = product.find("p", class_="price_color").text.strip()

        # Star rating
        rating = product.find("p", class_="star-rating")
        star_rating = rating["class"][1]

        # Availability
        availability = product.find(
            "p", class_="instock availability"
        ).text.strip()

        # Category
        # Open individual book page to get category
        book_url = "https://books.toscrape.com/catalogue/" + product.h3.a["href"]

        book_response = requests.get(book_url)
        book_soup = BeautifulSoup(book_response.text, "html.parser")

        breadcrumb = book_soup.find("ul", class_="breadcrumb")

        category = breadcrumb.find_all("li")[2].text.strip()

        # Store data
        books.append({
            "title": title,
            "price": price,
            "star_rating": star_rating,
            "availability": availability,
            "category": category
        })

# Convert to DataFrame
df = pd.DataFrame(books)

print("Number of books:", len(df))

df.head()

Number of books: 100


,title,price,star_rating,availability,category
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction
2,Soumission,Â£50.10,One,In stock,Fiction
3,Sharp Objects,Â£47.82,Four,In stock,Mystery
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History


In [2]:
df

,title,price,star_rating,availability,category
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction
2,Soumission,Â£50.10,One,In stock,Fiction
3,Sharp Objects,Â£47.82,Four,In stock,Mystery
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History
...,...,...,...,...,...
95,Lumberjanes Vol. 3: A Terrible Plan (Lumberjan...,Â£19.92,Two,In stock,Sequential Art
96,"Layered: Baking, Building, and Styling Spectac...",Â£40.11,One,In stock,Food and Drink
97,Judo: Seven Steps to Black Belt (an Introducto...,Â£53.90,Two,In stock,Add a comment
98,Join,Â£35.67,Five,In stock,Science Fiction


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         100 non-null    object
 1   price         100 non-null    object
 2   star_rating   100 non-null    object
 3   availability  100 non-null    object
 4   category      100 non-null    object
dtypes: object(5)
memory usage: 4.0+ KB


In [4]:

df["price_gdp"]=df["price"].str.replace("Â£","").astype("float64")

In [5]:
df

,title,price,star_rating,availability,category,price_gdp
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry,51.77
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction,53.74
2,Soumission,Â£50.10,One,In stock,Fiction,50.10
3,Sharp Objects,Â£47.82,Four,In stock,Mystery,47.82
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History,54.23
...,...,...,...,...,...,...
95,Lumberjanes Vol. 3: A Terrible Plan (Lumberjan...,Â£19.92,Two,In stock,Sequential Art,19.92
96,"Layered: Baking, Building, and Styling Spectac...",Â£40.11,One,In stock,Food and Drink,40.11
97,Judo: Seven Steps to Black Belt (an Introducto...,Â£53.90,Two,In stock,Add a comment,53.90
98,Join,Â£35.67,Five,In stock,Science Fiction,35.67


In [6]:
df["star_rating"].unique()

array(['Three', 'One', 'Four', 'Five', 'Two'], dtype=object)

In [7]:
df["rating"] = (
    df["star_rating"]
    .str.replace("Three", "3")
    .replace("One", "1")
    .replace("Four", "4")
    .replace("Five", "5")
    .replace("Two", "2")
    .astype(int)
)


In [8]:
df["availability"].unique()

array(['In stock'], dtype=object)

In [9]:
df["in_stock"]=df["availability"].map({'In stock':True})
df


,title,price,star_rating,availability,category,price_gdp,rating,in_stock
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry,51.77,3,True
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction,53.74,1,True
2,Soumission,Â£50.10,One,In stock,Fiction,50.10,1,True
3,Sharp Objects,Â£47.82,Four,In stock,Mystery,47.82,4,True
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History,54.23,5,True
...,...,...,...,...,...,...,...,...
95,Lumberjanes Vol. 3: A Terrible Plan (Lumberjan...,Â£19.92,Two,In stock,Sequential Art,19.92,2,True
96,"Layered: Baking, Building, and Styling Spectac...",Â£40.11,One,In stock,Food and Drink,40.11,1,True
97,Judo: Seven Steps to Black Belt (an Introducto...,Â£53.90,Two,In stock,Add a comment,53.90,2,True
98,Join,Â£35.67,Five,In stock,Science Fiction,35.67,5,True


In [10]:
df.isnull().sum()

,0
title,0
price,0
star_rating,0
availability,0
category,0
price_gdp,0
rating,0
in_stock,0


In [11]:
df.describe()

,price_gdp,rating
count,100.000000,100.000000
mean,34.560700,2.930000
std,14.638531,1.423149
min,10.160000,1.000000
25%,19.897500,2.000000
50%,34.775000,3.000000
75%,47.967500,4.000000
max,58.110000,5.000000


In [12]:
df["rating"] = df["rating"].fillna(df["rating"].mean())

In [13]:
df.isnull().sum()

,0
title,0
price,0
star_rating,0
availability,0
category,0
price_gdp,0
rating,0
in_stock,0


In [14]:
df.columns

Index(['title', 'price', 'star_rating', 'availability', 'category',
       'price_gdp', 'rating', 'in_stock'],
      dtype='object')

In [15]:
 df['price_inr']= df['price_gdp']*105.50

In [16]:
df

,title,price,star_rating,availability,category,price_gdp,rating,in_stock,price_inr
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry,51.77,3,True,5461.735
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction,53.74,1,True,5669.570
2,Soumission,Â£50.10,One,In stock,Fiction,50.10,1,True,5285.550
3,Sharp Objects,Â£47.82,Four,In stock,Mystery,47.82,4,True,5045.010
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History,54.23,5,True,5721.265
...,...,...,...,...,...,...,...,...,...
95,Lumberjanes Vol. 3: A Terrible Plan (Lumberjan...,Â£19.92,Two,In stock,Sequential Art,19.92,2,True,2101.560
96,"Layered: Baking, Building, and Styling Spectac...",Â£40.11,One,In stock,Food and Drink,40.11,1,True,4231.605
97,Judo: Seven Steps to Black Belt (an Introducto...,Â£53.90,Two,In stock,Add a comment,53.90,2,True,5686.450
98,Join,Â£35.67,Five,In stock,Science Fiction,35.67,5,True,3763.185


In [17]:
import sqlite3

conn = sqlite3.connect("books.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY,
    title TEXT,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
""")

conn.commit()

print("Tables created successfully")

Tables created successfully


In [18]:
df = df.rename(columns={"price_gdp": "price_gbp"})

In [19]:
import sqlite3

conn = sqlite3.connect("books.db")
cursor = conn.cursor()

# Get unique categories
categories = df["category"].unique()

for category in categories:
    cursor.execute(
        "INSERT OR IGNORE INTO categories (category_name) VALUES (?)",
        (category,)
    )

conn.commit()

In [20]:
for _, row in df.iterrows():

    cursor.execute(
        "SELECT category_id FROM categories WHERE category_name = ?",
        (row["category"],)
    )

    category_id = cursor.fetchone()[0]

    cursor.execute("""
        INSERT INTO books
        (title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        row["rating"],
        int(row["in_stock"]),
        category_id
    ))

conn.commit()

In [21]:
df.columns

Index(['title', 'price', 'star_rating', 'availability', 'category',
       'price_gbp', 'rating', 'in_stock', 'price_inr'],
      dtype='object')

In [22]:
query1 = """
SELECT title, rating
FROM books
WHERE rating = 5
"""

result1 = cursor.execute(query1).fetchall()

print(result1)

[('Sapiens: A Brief History of Humankind', 5), ('Set Me Free', 5), ("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 5), ('Rip it Up and Start Again', 5), ('Chase Me (Paris Nights #2)', 5), ('Black Dust', 5), ('Worlds Elsewhere: Journeys Around Shakespeareâ\x80\x99s Globe', 5), ('The Four Agreements: A Practical Guide to Personal Freedom', 5), ('The Elephant Tree', 5), ("Sophie's World", 5), ('Private Paris (Private #10)', 5), ('#HigherSelfie: Wake Up Your Life. Free Your Soul. Find Your Tribe.', 5), ('We Love You, Charlie Freeman', 5), ('Thirst', 5), ('The Inefficiency Assassin: Time Management Tactics for Working Smarter, Not Longer', 5), ("The Activist's Tao Te Ching: Ancient Advice for a Modern Revolution", 5), ('Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Princess Jellyfish 2-in-1 Omnibus #1)', 5), ('Princess Between Worlds (Wide-Awake Princess #5)', 5), ('Join', 5)]


In [23]:
query2 = """
SELECT title, price_gbp
FROM books
ORDER BY price_gbp DESC
"""

result2 = cursor.execute(query2).fetchall()

print(result2)

[('The Death of Humanity: and the Case for Life', 58.11), ('Slow States of Collapse: Poems', 57.31), ('Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991', 57.25), ('The Past Never Ends', 56.5), ('The Pioneer Woman Cooks: Dinnertime: Comfort Classics, Freezer Food, 16-Minute Meals, and Other Delicious Ways to Solve Supper!', 56.41), ('Masks and Shadows', 56.4), ('The Secret of Dreadwillow Carse', 56.13), ('The Electric Pencil: Drawings from Inside State Hospital No. 3', 56.06), ('Birdsong: A Story in Pictures', 54.64), ('Sapiens: A Brief History of Humankind', 54.23), ('The Murder That Never Was (Forensic Instincts #5)', 54.11), ('Judo: Seven Steps to Black Belt (an Introductory Guide for Beginners)', 53.9), ('Tipping the Velvet', 53.74), ('Aladdin and His Wonderful Lamp', 53.13), ("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 52.29), ('Behind Closed Doors', 52.22), ('The Black Maria', 52.15), ('A Light in the Attic', 51.77), ('Libertarianis

In [24]:
query3 = """
SELECT title, price_gbp
FROM books
ORDER BY price_gbp DESC
LIMIT 10
"""

result3 = cursor.execute(query3).fetchall()

print(result3)

[('The Death of Humanity: and the Case for Life', 58.11), ('Slow States of Collapse: Poems', 57.31), ('Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991', 57.25), ('The Past Never Ends', 56.5), ('The Pioneer Woman Cooks: Dinnertime: Comfort Classics, Freezer Food, 16-Minute Meals, and Other Delicious Ways to Solve Supper!', 56.41), ('Masks and Shadows', 56.4), ('The Secret of Dreadwillow Carse', 56.13), ('The Electric Pencil: Drawings from Inside State Hospital No. 3', 56.06), ('Birdsong: A Story in Pictures', 54.64), ('Sapiens: A Brief History of Humankind', 54.23)]


In [25]:
query4 = """
SELECT DISTINCT category_name
FROM categories
"""

result4 = cursor.execute(query4).fetchall()

print(result4)

[('Add a comment',), ('Art',), ('Business',), ('Childrens',), ('Contemporary',), ('Default',), ('Fantasy',), ('Fiction',), ('Food and Drink',), ('Health',), ('Historical Fiction',), ('History',), ('Horror',), ('Music',), ('Mystery',), ('New Adult',), ('Nonfiction',), ('Philosophy',), ('Poetry',), ('Politics',), ('Romance',), ('Science',), ('Science Fiction',), ('Self Help',), ('Sequential Art',), ('Spirituality',), ('Thriller',), ('Travel',), ('Young Adult',)]


In [26]:
query5 = """
SELECT title, price_gbp
FROM books
WHERE price_gbp BETWEEN 20 AND 30
"""

result5 = cursor.execute(query5).fetchall()

print(result5)

[('The Requiem Red', 22.65), ('The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics', 22.6), ("Shakespeare's Sonnets", 20.66), ('Olio', 23.88), ('Chase Me (Paris Nights #2)', 25.27), ("America's Cradle of Quarterbacks: Western Pennsylvania's Football Factory from Johnny Unitas to Joe Montana", 22.5), ('The Elephant Tree', 23.82), ('Reasons to Stay Alive', 26.41), ('#HigherSelfie: Wake Up Your Life. Free Your Soul. Find Your Tribe.', 23.11), ('Unbound: How Eight Technologies Made Us Human, Transformed Society, and Brought Our World to the Brink', 25.52), ('The Mindfulness and Acceptance Workbook for Anxiety: A Guide to Breaking Free from Anxiety, Phobias, and Worry Using Acceptance and Commitment Therapy', 23.89), ('The Inefficiency Assassin: Time Management Tactics for Working Smarter, Not Longer', 20.59), ('Saga, Volume 6 (Saga (Collected Editions) #6)', 25.02), ('In the Country We Love: My Family Divided', 22.0)]


In [27]:
query6 = """
SELECT
    books.title,
    books.rating,
    categories.category_name
FROM books
JOIN categories
ON books.category_id = categories.category_id
ORDER BY books.rating DESC
LIMIT 10
"""

result6 = cursor.execute(query6).fetchall()

print(result6)

[('Sapiens: A Brief History of Humankind', 5, 'History'), ('Set Me Free', 5, 'Young Adult'), ("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 5, 'Sequential Art'), ('Rip it Up and Start Again', 5, 'Music'), ('Chase Me (Paris Nights #2)', 5, 'Romance'), ('Black Dust', 5, 'Romance'), ('Worlds Elsewhere: Journeys Around Shakespeareâ\x80\x99s Globe', 5, 'Nonfiction'), ('The Four Agreements: A Practical Guide to Personal Freedom', 5, 'Spirituality'), ('The Elephant Tree', 5, 'Thriller'), ("Sophie's World", 5, 'Philosophy')]


In [28]:
queries = [
    ("Query 1 - SELECT WHERE", query1, result1),
    ("Query 2 - ORDER BY", query2, result2),
    ("Query 3 - LIMIT", query3, result3),
    ("Query 4 - DISTINCT", query4, result4),
    ("Query 5 - BETWEEN", query5, result5),
    ("Query 6 - JOIN", query6, result6)
]

with open("query_outputs.txt", "w", encoding="utf-8") as f:

    for name, query, output in queries:

        f.write("=" * 60 + "\n")
        f.write(name + "\n")
        f.write("=" * 60 + "\n")

        f.write("SQL Query:\n")
        f.write(query.strip() + "\n\n")

        f.write("Output:\n")
        for row in output:
            f.write(str(row) + "\n")

        f.write("\n")

In [29]:
conn.close()